# Generate prompt instances

This notebook builds the prompt-instance dataset from the finalized clean base examples.

It uses the redesigned prompt pipeline, including:
- sampled bounded openers
- multiple bounded layouts
- naturalistic unbounded prompt surfaces
- richer noise libraries
- structured distraction metadata
- updated prompt-instance validation

Outputs:
- `data/prompts/prompt_instances.jsonl`
- `data/prompts/prompt_instances.json`
- `data/prompts/prompt_instance_summary.json`
- `data/prompts/prompt_instance_validation.json`
- `data/prompts/prompt_preview_samples.json`
- `data/specs/prompt_design_spec.json`

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions')

## 1) Imports

We import the updated prompt-generation and validation pipeline from `src/`.

In [2]:
from src.prompt_instance_generation import (
    load_jsonl,
    save_jsonl,
    save_json,
    build_all_prompt_instances,
    build_prompt_summary,
    build_prompt_preview_samples,
)

from src.prompt_instance_validation import validate_prompt_instances

from src.prompt_builder import (
    build_prompt_previews,
    export_prompt_design_spec,
)

## 2) Input and output paths

In [3]:
BASE_INPUT_PATH = PROJECT_ROOT / "data" / "base" / "base_examples.jsonl"

PROMPT_JSONL_OUTPUT_PATH = PROJECT_ROOT / "data" / "prompts" / "prompt_instances.jsonl"
PROMPT_JSON_OUTPUT_PATH = PROJECT_ROOT / "data" / "prompts" / "prompt_instances.json"
PROMPT_SUMMARY_OUTPUT_PATH = PROJECT_ROOT / "data" / "prompts" / "prompt_instance_summary.json"
PROMPT_VALIDATION_OUTPUT_PATH = PROJECT_ROOT / "data" / "prompts" / "prompt_instance_validation.json"
PROMPT_PREVIEW_OUTPUT_PATH = PROJECT_ROOT / "data" / "prompts" / "prompt_preview_samples.json"

PROMPT_DESIGN_SPEC_OUTPUT_PATH = PROJECT_ROOT / "data" / "specs" / "prompt_design_spec.json"

BASE_INPUT_PATH, PROMPT_JSONL_OUTPUT_PATH

(WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions/data/base/base_examples.jsonl'),
 WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions/data/prompts/prompt_instances.jsonl'))

## 3) Load finalized base examples

These are the clean benchmark records from the earlier dataset-building stage.

In [4]:
base_records = load_jsonl(str(BASE_INPUT_PATH))

len(base_records), base_records[0].keys() if base_records else None

(250,
 dict_keys(['example_id', 'gold_output', 'input_data', 'instruction', 'metadata', 'task_name', 'template_name']))

In [5]:
if not base_records:
    raise ValueError("No base records found. Check data/base/base_examples.jsonl")

task_counts = {}
for record in base_records:
    task_counts[record["task_name"]] = task_counts.get(record["task_name"], 0) + 1

task_counts

{'extractive_qa': 50,
 'information_extraction': 50,
 'multi_label_classification': 50,
 'rule_based_transformation': 50,
 'single_label_classification': 50}

## 4) Quick inspection of one base example

In [6]:
example_preview = base_records[0]
print(json.dumps(example_preview, indent=2, ensure_ascii=False))

{
  "example_id": "qa_200",
  "gold_output": {
    "answer": "Milan"
  },
  "input_data": {
    "passage": "Alice Smith left Rome on 2024-04-02, arrived in Milan on 2024-03-21 for a training session, and later continued to Turin for a short meeting.",
    "question": "Where did Alice Smith arrive on 2024-03-21?"
  },
  "instruction": "Return the exact words from the passage that answer the question.",
  "metadata": {
    "answer_field": "location"
  },
  "task_name": "extractive_qa",
  "template_name": "schedule_passage_rich"
}


## 5) Build all prompt instances

For each base example:
- 2 regimes: bounded, unbounded
- 8 prompt types:
  - clean
  - irrelevant_prefix
  - irrelevant_suffix
  - instruction_in_the_middle
  - conflicting_instruction
  - negation_distraction
  - style_distraction
  - length_stress

Expected total if there are 250 base examples:
- 250 × 2 × 8 = 4000 prompt instances

In [7]:
prompt_records = build_all_prompt_instances(base_records)

len(prompt_records)

4000

In [8]:
expected_total = len(base_records) * 2 * 8
print("Expected total:", expected_total)
print("Observed total:", len(prompt_records))

Expected total: 4000
Observed total: 4000


## 6) Inspect a few generated prompt records

In [9]:
for idx in range(3):
    print("=" * 100)
    print(json.dumps(prompt_records[idx], indent=2, ensure_ascii=False))

{
  "prompt_id": "qa_200__bounded__clean",
  "base_example_id": "qa_200",
  "task_name": "extractive_qa",
  "distraction_type": "clean",
  "distraction_subtype": null,
  "distraction_variant_id": null,
  "regime": "bounded",
  "is_clean": true,
  "prompt_text": "Please continue by handling the request shown below.\n\nInput material\n\n<INPUT>\nPassage:\nAlice Smith left Rome on 2024-04-02, arrived in Milan on 2024-03-21 for a training session, and later continued to Turin for a short meeting.\n\nQuestion:\nWhere did Alice Smith arrive on 2024-03-21?\n</INPUT>\n\nInstruction\n\n<TASK>\nReturn the exact words from the passage that answer the question.\n</TASK>",
  "gold_output": {
    "answer": "Milan"
  },
  "source_instruction": "Return the exact words from the passage that answer the question.",
  "source_template_name": "schedule_passage_rich",
  "prompt_surface_type": "bounded_tagged",
  "surface_id": null,
  "surface_name": null,
  "opener_id": "bo_034",
  "opener_text": "Please co

## 7) Inspect prompt text only for selected examples

In [10]:
sample_indices = [0, 1, 2, 20, 40]

for idx in sample_indices:
    print("=" * 120)
    print("prompt_id:", prompt_records[idx]["prompt_id"])
    print("task_name:", prompt_records[idx]["task_name"])
    print("regime:", prompt_records[idx]["regime"])
    print("distraction_type:", prompt_records[idx]["distraction_type"])
    print("distraction_subtype:", prompt_records[idx].get("distraction_subtype"))
    print("-" * 120)
    print(prompt_records[idx]["prompt_text"])
    print()

prompt_id: qa_200__bounded__clean
task_name: extractive_qa
regime: bounded
distraction_type: clean
distraction_subtype: None
------------------------------------------------------------------------------------------------------------------------
Please continue by handling the request shown below.

Input material

<INPUT>
Passage:
Alice Smith left Rome on 2024-04-02, arrived in Milan on 2024-03-21 for a training session, and later continued to Turin for a short meeting.

Question:
Where did Alice Smith arrive on 2024-03-21?
</INPUT>

Instruction

<TASK>
Return the exact words from the passage that answer the question.
</TASK>

prompt_id: qa_200__bounded__irrelevant_prefix
task_name: extractive_qa
regime: bounded
distraction_type: irrelevant_prefix
distraction_subtype: short_prefix_noise
------------------------------------------------------------------------------------------------------------------------
The room was quiet except for the slow tapping of rain against the outer glass. O

## 8) Build dataset summary

In [11]:
prompt_summary = build_prompt_summary(prompt_records)
print(json.dumps(prompt_summary, indent=2, ensure_ascii=False))

{
  "total_prompt_instances": 4000,
  "counts_by_task": {
    "extractive_qa": 800,
    "information_extraction": 800,
    "multi_label_classification": 800,
    "rule_based_transformation": 800,
    "single_label_classification": 800
  },
  "counts_by_regime": {
    "bounded": 2000,
    "unbounded": 2000
  },
  "counts_by_distraction_type": {
    "clean": 500,
    "irrelevant_prefix": 500,
    "irrelevant_suffix": 500,
    "instruction_in_the_middle": 500,
    "conflicting_instruction": 500,
    "negation_distraction": 500,
    "style_distraction": 500,
    "length_stress": 500
  },
  "counts_by_distraction_subtype": {
    "short_prefix_noise": 500,
    "short_suffix_noise": 500,
    "middle_burial": 500,
    "conflicting_task_substitution": 77,
    "direct_negation": 124,
    "social_post": 50,
    "mixed_code_and_comments": 79,
    "softened_counterinstruction": 100,
    "pirate_persona": 50,
    "email_thread_fragment": 55,
    "conflicting_format_override": 69,
    "paraphrastic_n

## 9) Validate prompt instances

This uses the updated validator that understands:
- richer prompt metadata
- bounded vs unbounded surface differences
- distraction subtypes
- structured noise/style/conflict metadata

In [12]:
prompt_validation = validate_prompt_instances(
    prompt_records,
    expected_total=len(base_records) * 2 * 8,
    expected_base_examples=len(base_records),
    expected_per_base_example=16,
)

print("is_valid:", prompt_validation["is_valid"])
print("num_records_with_issues:", prompt_validation["num_records_with_issues"])
print("dataset_level_issues:", prompt_validation["dataset_level_issues"])

is_valid: True
num_records_with_issues: 0
dataset_level_issues: []


In [13]:
print(json.dumps(prompt_validation, indent=2, ensure_ascii=False))

{
  "is_valid": true,
  "total_prompt_instances": 4000,
  "unique_base_example_ids": 250,
  "num_records_with_issues": 0,
  "record_issues": {},
  "dataset_level_issues": [],
  "issue_counts": {},
  "counts_by_task": {
    "extractive_qa": 800,
    "information_extraction": 800,
    "multi_label_classification": 800,
    "rule_based_transformation": 800,
    "single_label_classification": 800
  },
  "counts_by_regime": {
    "bounded": 2000,
    "unbounded": 2000
  },
  "counts_by_distraction_type": {
    "clean": 500,
    "irrelevant_prefix": 500,
    "irrelevant_suffix": 500,
    "instruction_in_the_middle": 500,
    "conflicting_instruction": 500,
    "negation_distraction": 500,
    "style_distraction": 500,
    "length_stress": 500
  },
  "counts_by_distraction_subtype": {
    "short_prefix_noise": 500,
    "short_suffix_noise": 500,
    "middle_burial": 500,
    "conflicting_task_substitution": 77,
    "direct_negation": 124,
    "social_post": 50,
    "mixed_code_and_comments": 

## 10) Build preview samples for manual inspection

We generate a smaller preview subset so it is easier to inspect coverage and prompt quality.

In [14]:
prompt_previews = build_prompt_previews(
    records=base_records,
    examples_per_task=2,
)

len(prompt_previews)

160

In [15]:
for idx in range(min(5, len(prompt_previews))):
    print("=" * 120)
    print(json.dumps(prompt_previews[idx], indent=2, ensure_ascii=False))

{
  "example_id": "qa_200",
  "task_name": "extractive_qa",
  "regime": "bounded",
  "prompt_type": "clean",
  "prompt_text": "Please review the marked request below.\n\nRequest section\n\n<TASK>\nReturn the exact words from the passage that answer the question.\n</TASK>\n\nProvided material\n\n<INPUT>\nPassage:\nAlice Smith left Rome on 2024-04-02, arrived in Milan on 2024-03-21 for a training session, and later continued to Turin for a short meeting.\n\nQuestion:\nWhere did Alice Smith arrive on 2024-03-21?\n</INPUT>",
  "gold_output": {
    "answer": "Milan"
  },
  "source_instruction": "Return the exact words from the passage that answer the question.",
  "source_template_name": "schedule_passage_rich",
  "prompt_surface_type": "bounded_tagged",
  "surface_id": null,
  "surface_name": null,
  "opener_id": "bo_001",
  "opener_text": "Please review the marked request below.",
  "layout_id": "bl_002",
  "layout_name": "task_then_input_labeled"
}
{
  "example_id": "qa_200",
  "task_nam

## 11) Build compact preview samples directly from the full prompt-instance set

This is useful if you want one representative record per `(task, regime, distraction_type)` condition.

In [16]:
prompt_preview_samples = build_prompt_preview_samples(
    prompt_records,
    n_per_condition=1,
)

len(prompt_preview_samples)

80

In [17]:
for idx in range(min(5, len(prompt_preview_samples))):
    print("=" * 120)
    print(json.dumps(prompt_preview_samples[idx], indent=2, ensure_ascii=False))

{
  "prompt_id": "qa_200__bounded__clean",
  "base_example_id": "qa_200",
  "task_name": "extractive_qa",
  "distraction_type": "clean",
  "distraction_subtype": null,
  "distraction_variant_id": null,
  "regime": "bounded",
  "is_clean": true,
  "prompt_text": "Please continue by handling the request shown below.\n\nInput material\n\n<INPUT>\nPassage:\nAlice Smith left Rome on 2024-04-02, arrived in Milan on 2024-03-21 for a training session, and later continued to Turin for a short meeting.\n\nQuestion:\nWhere did Alice Smith arrive on 2024-03-21?\n</INPUT>\n\nInstruction\n\n<TASK>\nReturn the exact words from the passage that answer the question.\n</TASK>",
  "gold_output": {
    "answer": "Milan"
  },
  "source_instruction": "Return the exact words from the passage that answer the question.",
  "source_template_name": "schedule_passage_rich",
  "prompt_surface_type": "bounded_tagged",
  "surface_id": null,
  "surface_name": null,
  "opener_id": "bo_034",
  "opener_text": "Please co

## 12) Export prompt design specification

This writes a machine-readable snapshot of the current prompt design:
- openers
- layouts
- surfaces
- noise libraries
- style distractions
- conflicting instructions
- negation libraries

In [18]:
export_prompt_design_spec(str(PROMPT_DESIGN_SPEC_OUTPUT_PATH))
print("Saved:", PROMPT_DESIGN_SPEC_OUTPUT_PATH)

Saved: C:\code\testing\LLMs_Robustness_Under_Distractions\data\specs\prompt_design_spec.json


## 13) Save full prompt-instance outputs

In [19]:
save_jsonl(prompt_records, str(PROMPT_JSONL_OUTPUT_PATH))
save_json(prompt_records, str(PROMPT_JSON_OUTPUT_PATH))
save_json(prompt_summary, str(PROMPT_SUMMARY_OUTPUT_PATH))
save_json(prompt_validation, str(PROMPT_VALIDATION_OUTPUT_PATH))
save_json(prompt_preview_samples, str(PROMPT_PREVIEW_OUTPUT_PATH))

print("Saved JSONL:", PROMPT_JSONL_OUTPUT_PATH)
print("Saved JSON :", PROMPT_JSON_OUTPUT_PATH)
print("Saved summary:", PROMPT_SUMMARY_OUTPUT_PATH)
print("Saved validation:", PROMPT_VALIDATION_OUTPUT_PATH)
print("Saved preview samples:", PROMPT_PREVIEW_OUTPUT_PATH)

Saved JSONL: C:\code\testing\LLMs_Robustness_Under_Distractions\data\prompts\prompt_instances.jsonl
Saved JSON : C:\code\testing\LLMs_Robustness_Under_Distractions\data\prompts\prompt_instances.json
Saved summary: C:\code\testing\LLMs_Robustness_Under_Distractions\data\prompts\prompt_instance_summary.json
Saved validation: C:\code\testing\LLMs_Robustness_Under_Distractions\data\prompts\prompt_instance_validation.json
Saved preview samples: C:\code\testing\LLMs_Robustness_Under_Distractions\data\prompts\prompt_preview_samples.json


We generated all prompt instances for Phase 5 by taking each base example and rendering it in two regimes (bounded and unbounded) across eight prompt conditions (one clean condition and seven distracted conditions). Each prompt instance was stored with a prompt ID, base-example ID, task label, distraction type, regime, clean/distracted flag, prompt text, and gold output. Before moving on to model inference, a preview sample of prompts was inspected manually to verify that the wrappers and distraction templates were rendered correctly.

## 14) Final sanity checks

In [20]:
assert len(prompt_records) == len(base_records) * 16, "Unexpected number of prompt instances"
assert prompt_summary["total_prompt_instances"] == len(prompt_records), "Summary total mismatch"
assert Path(PROMPT_JSONL_OUTPUT_PATH).exists(), "Prompt JSONL was not written"
assert Path(PROMPT_SUMMARY_OUTPUT_PATH).exists(), "Prompt summary was not written"
assert Path(PROMPT_VALIDATION_OUTPUT_PATH).exists(), "Prompt validation was not written"

print("All sanity checks passed.")

All sanity checks passed.


## 15) Debug: inspect issue examples if validation fails

In [21]:
if not prompt_validation["is_valid"]:
    print("Validation failed. Showing first few problematic records:\n")
    shown = 0
    for prompt_id, issues in prompt_validation["record_issues"].items():
        print("=" * 100)
        print("prompt_id:", prompt_id)
        print("issues:", issues)
        matching = [r for r in prompt_records if r["prompt_id"] == prompt_id]
        if matching:
            print(json.dumps(matching[0], indent=2, ensure_ascii=False))
        shown += 1
        if shown >= 3:
            break
else:
    print("Validation passed with no record-level issues.")

Validation passed with no record-level issues.
